# Facial Keypoints Detection — v4ctrl — seed-noise control: exact v2 configuration, seed 7

The one re-run of the v2 configuration, changing only the seed. Only the seed differs (7 vs 42). |v2 - control| is the noise band every v4 comparison is judged against.

Same 88% / 12% split as v2 (`random_state=42`), so the held-out RMSE is directly comparable to v2's 2.4385px. One log line per epoch goes to `logs/v4ctrl.log`; the best weights are checkpointed on every improvement to `checkpoints/v4ctrl.pt`. No figures are drawn during training: every epoch appends statistics (loss split, held-out RMSE overall and per coordinate, centre-pull slope, softmax entropy, temperature, gradient norm, learning rate) to `results/v4ctrl_history.json`; they are plotted in the post-training notebook.

In [1]:
import json, time
import torch
import fkd_v4_common as C
print('device:', C.device(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
data = C.load_data('data')
print('train rows', len(data['idx_tr']), '| held-out rows', len(data['idx_va']))

device: cuda NVIDIA L4


train rows 6203 | held-out rows 846


In [2]:
RUN = 'v4ctrl'
CFG = {'epochs': 0,
 'a_max': 8,
 'a_patience': 99,
 'batch': 256,
 'lr': 0.001,
 'seed': 7,
 'opt': 'adam',
 'H': 24,
 'sigma': 1.0,
 'mapping': 'v2',
 'heatmap_loss': 'mse',
 'hm_frac': 0,
 'aug': {'affine': False, 'photometric': False, 'cutout': False},
 'b_phases': [{'epochs': 20, 'lr': 0.0005, 'sched': 'plateau', 'plateau_patience': 3, 'patience': 5},
              {'epochs': 250, 'lr': 0.0003, 'sched': 'plateau', 'plateau_patience': 6, 'patience': 30}]}
model = C.SingleStageModel(H=24, n_deconv=1, beta=1.0, learn_beta=False, mapping="v2")
print(sum(p.numel() for p in model.parameters()), 'parameters')

622032 parameters


## Training (Stage A heatmap warm-up to plateau, then Stage B) — statistics only, no figures

In [3]:
t0 = time.time()
model, history, best = C.train(RUN, model, data, CFG)
print(f'best held-out RMSE {best:.4f}px | wall {time.time()-t0:.0f}s')

2026-10-08T12:28:26Z [v4ctrl] start | params=622,032 | cfg={"epochs": 0, "a_max": 8, "a_patience": 99, "batch": 256, "lr": 0.001, "seed": 7, "opt": "adam", "H": 24, "sigma": 1.0, "mapping": "v2", "heatmap_loss": "mse", "hm_frac": 0, "aug": {"affine": false, "photometric": false, "cutout": false}, "b_phases": [{"epochs": 20, "lr": 0.0005, "sched": "plateau", "plateau_patience": 3, "patience": 5}, {"epochs": 250, "lr": 0.0003, "sched": "plateau", "plateau_patience": 6, "patience": 30}]}


fkd_v4_common.py:510: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  k = len(idx); s["loss"] += float(loss) * k; s["l1"] += float(l1) * k; s["hm"] += float(ht) * k; s["n"] += k


2026-10-08T12:28:59Z [v4ctrl:A] epoch   0 train_hm 0.07357 val_hm 0.00770 val_rmse 18.0695 beta 1.00 29.8s


2026-10-08T12:29:04Z [v4ctrl:A] epoch   1 train_hm 0.00635 val_hm 0.00552 val_rmse 18.0598 beta 1.00 5.1s


2026-10-08T12:29:09Z [v4ctrl:A] epoch   2 train_hm 0.00535 val_hm 0.00523 val_rmse 18.0513 beta 1.00 4.7s


...

2026-10-08T12:41:22Z [v4ctrl:B2] epoch 160 loss 1.1043 (l1 1.1043 hm 0) val_rmse 2.4632 best 2.4601 slope 0.574 lr 2.3e-06 beta 1.00 gnorm 7.47 4.5s


2026-10-08T12:41:26Z [v4ctrl:B2] epoch 161 loss 1.1066 (l1 1.1066 hm 0) val_rmse 2.4630 best 2.4601 slope 0.573 lr 2.3e-06 beta 1.00 gnorm 11.58 4.5s


2026-10-08T12:41:26Z [v4ctrl:B2] early stop at epoch 161 (no improvement for 30 epochs)


2026-10-08T12:41:26Z [v4ctrl] done | best held-out RMSE 2.4601 px | epochs used 162


best held-out RMSE 2.4601px | wall 781s


## Held-out evaluation: decoders x flip test, per-keypoint RMSE, centre-pull slopes

In [4]:
res = C.evaluate_and_save(RUN, model, data, CFG, decoders=('softargmax', 'argmax_shift', 'dark'))
for k, v in res.items():
    if k.startswith('rmse') or k in ('shrinkage_slope_mean', 'beta_final'):
        print(f'{k:28s} {v}')
print('v2 reference 2.4385 px | target 1.9 px')

rmse_softargmax              2.460099697113037
rmse_softargmax_tta          2.386847734451294
rmse_argmax_shift            12.07730484008789
rmse_argmax_shift_tta        10.310613632202148
rmse_dark                    17.210737228393555
rmse_dark_tta                15.177837371826172
shrinkage_slope_mean         0.5730079672182088
beta_final                   1.0
v2 reference 2.4385 px | target 1.9 px


## Test-set submission (written, not submitted)
Decoder chosen by held-out RMSE with flip test among the three evaluated above (a selection over three options, stated as such).

In [5]:
cands = {d: res[f'rmse_{d}_tta'] for d in ('softargmax', 'argmax_shift', 'dark')}
best_dec = min(cands, key=cands.get)
n = C.write_submission(RUN, model, data, CFG, tta=True, decoder=best_dec)
print('decoder', best_dec, '| held-out', round(cands[best_dec], 4), '| rows', n)

decoder softargmax | held-out 2.3868 | rows 27124
